# CAPI Inter-Rater Reliability — Quadratic Weighted Cohen's Kappa

This notebook computes the **inter-rater reliability** for the human evaluation of chatbot responses using **Cohen's Kappa**. Three variants of Cohen's Kappa (unweighted, linear weighted, and quadratic weighted) are calculated for comparison; however, **Quadratic Weighted Kappa (QWK)** is adopted as the primary reliability measure because the response labels are ordinal.

* **Unweighted Cohen's Kappa** — treats all disagreements equally.
* **Linear Weighted Kappa** — assigns linearly increasing penalties to disagreements.
* **Quadratic Weighted Kappa (QWK)** — assigns quadratically increasing penalties, making it the most appropriate measure for ordinal ratings.

**Measurement decisions**:

* Two independent human raters, each with at least **three years of experience in the informatics domain**, evaluated every chatbot response.
* Responses were assigned one of three ordinal labels: **Incorrect (0)**, **Partially Correct (1)**, or **Correct (2)**.
* Inter-rater agreement was computed for all Kappa variants, while **Quadratic Weighted Kappa** was reported as the primary reliability statistic.

Requires: `pandas`, `scikit-learn` (`pip install pandas scikit-learn`).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
import warnings, os
warnings.filterwarnings("ignore")


# Configuration



In [ ]:
KEY_COL       = "Query"
ABLATION_COLS = [
    "RAG+PA_Top1", # LLM + RAG + Prompt Adaptation + Top-1
    "RAG+PA_Top3",   # LLM + RAG + Prompt Adaptation + Top-3
    "RAG+PA_Top5",   # LLM + RAG + Prompt Adaptation + Top-5
    "RAG_Top1",  # LLM + RAG Top-1, without Prompt Adaptation
    "LLM_Only",       # LLM Only, without RAG, without Prompt Adaptation
]

OUT_DIR       = "."
SCALE_ORDER   = ["Incorrect", "Partially Correct", "Correct"]
SCALE_SCORE   = {label: i for i, label in enumerate(SCALE_ORDER)}
KAPPA_COLS    = ["Cohen_Kappa","Weighted_Kappa","Scotts_Pi","Fleiss_Kappa","Krippendorff_Alpha"]
KAPPA_LABELS  = ["Cohen κ","Weighted κ","Scott π","Fleiss κ","Krippendorff α"]

# Theme Colour
COLORS_DIST   = {"Correct": "#2ecc71", "Partially Correct": "#f39c12", "Incorrect": "#e74c3c"}
KAPPA_PALETTE = ["#3498db","#9b59b6","#e67e22","#1abc9c","#e74c3c"]
BG_COLOR      = "#f8f9fa"
SPINE_COLOR   = "#dddddd"


# KAPPA & Distribution

In [ ]:
def encode(series):
    return series.map(SCALE_SCORE).values.astype(float)

def scotts_pi(y1, y2):
    cats = np.unique(np.concatenate([y1, y2]))
    n = len(y1)
    Po = np.mean(y1 == y2)
    Pe = sum(((np.sum(y1==c)+np.sum(y2==c))/(2*n))**2 for c in cats)
    return float("nan") if Pe==1 else (Po-Pe)/(1-Pe)

In [ ]:
def fleiss_kappa(y1, y2):
    cats = np.unique(np.concatenate([y1, y2]))
    n, N = len(y1), 2
    cat_idx = {c:i for i,c in enumerate(cats)}
    mat = np.zeros((n, len(cats)))
    for i,(a,b) in enumerate(zip(y1,y2)):
        mat[i,cat_idx[a]]+=1; mat[i,cat_idx[b]]+=1
    P_bar = np.mean([(np.sum(mat[i]**2)-N)/(N*(N-1)) for i in range(n)])
    p_j   = mat.sum(axis=0)/(n*N)
    Pe    = np.sum(p_j**2)
    return float("nan") if Pe==1 else (P_bar-Pe)/(1-Pe)

In [ ]:
def krippendorff_alpha_ordinal(y1, y2):
    data   = np.array([y1,y2], dtype=float)
    values = np.unique(data[~np.isnan(data)])
    if len(values)<2: return float("nan")
    val_idx = {v:i for i,v in enumerate(values)}
    n_v = len(values)
    coincidence = np.zeros((n_v, n_v))
    for col in range(data.shape[1]):
        valid = data[:,col][~np.isnan(data[:,col])]
        if len(valid)<2: continue
        for vi in valid:
            for vj in valid:
                coincidence[val_idx[vi],val_idx[vj]] += 1/(len(valid)-1)
    n_total = coincidence.sum()
    n_k     = coincidence.sum(axis=1)
    def d_ord(k,l):
        lo,hi = min(k,l),max(k,l)
        return (n_k[lo:hi+1].sum()-(n_k[lo]+n_k[hi])/2)**2
    Do = sum(coincidence[k,l]*d_ord(k,l) for k in range(n_v) for l in range(n_v))
    De = sum(n_k[k]*n_k[l]*d_ord(k,l) for k in range(n_v) for l in range(n_v))
    if De==0: return float("nan")
    return 1 - Do/(De/(n_total-1))

In [ ]:
def compute_all(col, r1, r2):
    common = r1.index.intersection(r2.index)
    y1r, y2r = r1.loc[common,col], r2.loc[common,col]
    valid = y1r.isin(SCALE_ORDER) & y2r.isin(SCALE_ORDER)
    y1r,y2r = y1r[valid],y2r[valid]
    y1,y2   = encode(y1r), encode(y2r)
    n = len(y1)
    if n<2: return {k:float("nan") for k in ["n","Pct_Agreement"]+KAPPA_COLS+
                    ["Mean_Score_R1","Mean_Score_R2","Mean_Score_Avg"]}
    return {
        "n": n,
        "Pct_Agreement"      : round(np.mean(y1==y2)*100,1),
        "Cohen_Kappa"        : round(cohen_kappa_score(y1,y2),4),
        "Weighted_Kappa"     : round(cohen_kappa_score(y1,y2,weights="quadratic"),4),
        "Scotts_Pi"          : round(scotts_pi(y1,y2),4),
        "Fleiss_Kappa"       : round(fleiss_kappa(y1,y2),4),
        "Krippendorff_Alpha" : round(krippendorff_alpha_ordinal(y1,y2),4),
        "Mean_Score_R1"      : round(float(np.mean(y1)),4),
        "Mean_Score_R2"      : round(float(np.mean(y2)),4),
        "Mean_Score_Avg"     : round(float((np.mean(y1)+np.mean(y2))/2),4),
    }

In [ ]:
def label_dist(col, r1, r2):
    combined = pd.concat([r1[col],r2[col]],ignore_index=True)
    combined = combined[combined.isin(SCALE_ORDER)]
    total    = len(combined)
    return {label: round(combined.eq(label).sum()/total*100,1) for label in SCALE_ORDER}

In [ ]:
def interpret(k):
    if np.isnan(k): return "N/A"
    if k < 0.00:   return "Poor"
    if k < 0.20:   return "Slight"
    if k < 0.40:   return "Fair"
    if k < 0.60:   return "Moderate"
    if k < 0.80:   return "Substantial"
    return "Almost Perfect"

In [ ]:
import pandas as pd
import csv

from pathlib import Path

# Repo root, resolved relative to this file — works regardless of where you run it from.
ROOT = Path(__file__).resolve().parents[1]   # notebook: Path.cwd().parents[0]
DATA = ROOT / "03_Data"

r1 = pd.read_csv(DATA / "RecapRater2.csv")
r2 = pd.read_csv(DATA / "RecapRater1.csv")


rename_mapping = {
    'Top3': 'RAG+PA_Top3',
    'Top1': 'RAG+PA_Top1',
    'Top5': 'RAG+PA_Top5',
    'LLMOnly': 'LLM_Only',
    'LLM+RAG': 'RAG_Top1'
}

r1 = r1.rename(columns=rename_mapping)
r2 = r2.rename(columns=rename_mapping)

# Re-calculate 'df' which contains 'Weighted_Kappa'
rows = []
for col in ABLATION_COLS:
    res = compute_all(col, r1, r2)
    rows.append({"Ablation": col, **res})
df = pd.DataFrame(rows)

# Extract 'Avg Kappa' (which corresponds to 'Weighted_Kappa')
kappa_avg = df["Weighted_Kappa"].values

# Create the requested DataFrame
avg_kappa_table = pd.DataFrame({
    "Method": ABLATION_COLS,
    "Weighted_Kappa": kappa_avg
})

display(avg_kappa_table)

# Analysis Disagreement

In [ ]:


def analyze_disagreement(col, r1, r2, scale_order=SCALE_ORDER):
    y1 = r1[col]
    y2 = r2[col]
    common = y1.index.intersection(y2.index)
    y1, y2 = y1.loc[common], y2.loc[common]

    # hanya baris yang keduanya berlabel valid (ada di SCALE_ORDER)
    valid = y1.isin(scale_order) & y2.isin(scale_order)
    y1v, y2v = y1[valid], y2[valid]

    n_total = len(y1v)
    agree_mask = (y1v == y2v)
    n_agree = int(agree_mask.sum())
    n_disagree = n_total - n_agree

    scale_map = {label: i for i, label in enumerate(scale_order)}
    dist = (y1v.map(scale_map) - y2v.map(scale_map)).abs()
    n_near_miss = int((dist == 1).sum())
    n_major = int((dist == 2).sum())

    return {
        "Method": col,
        "N": n_total,
        "Agree": n_agree,
        "Disagree": n_disagree,
        "Disagree_%": round(n_disagree / n_total * 100, 2) if n_total else float("nan"),
        "Near_miss (1-level)": n_near_miss,
        "Major (2-level)": n_major,
    }


def get_disagreement_items(col, r1, r2, scale_order=SCALE_ORDER):
    """Mengembalikan daftar item yang di-disagreement, untuk keperluan review/adjudication."""
    y1, y2 = r1[col], r2[col]
    common = y1.index.intersection(y2.index)
    y1, y2 = y1.loc[common], y2.loc[common]
    valid = y1.isin(scale_order) & y2.isin(scale_order)
    y1v, y2v = y1[valid], y2[valid]
    mask = y1v != y2v

    scale_map = {label: i for i, label in enumerate(scale_order)}
    out = pd.DataFrame({
        "Query": y1v.index[mask],
        "Method": col,
        "Rater1": y1v[mask].values,
        "Rater2": y2v[mask].values,
    })
    if len(out):
        out["Distance"] = (out["Rater1"].map(scale_map) - out["Rater2"].map(scale_map)).abs()
        out["Severity"] = out["Distance"].map({1: "Near-miss", 2: "Major"})
    return out


In [ ]:
#Summary Disagreement per Metode
summary = pd.DataFrame([analyze_disagreement(c, r1, r2) for c in ABLATION_COLS])
summary = summary.sort_values("Disagree_%").reset_index(drop=True)
summary

In [ ]:
#Graph- Disagreement
fig, ax = plt.subplots(figsize=(9, 5))

near_pct = summary["Near_miss (1-level)"] / summary["N"] * 100
major_pct = summary["Major (2-level)"] / summary["N"] * 100

ax.bar(summary["Method"], near_pct, label="Near-miss (1 level)", color="#f39c12")
ax.bar(summary["Method"], major_pct, bottom=near_pct, label="Major (2 level)", color="#e74c3c")

for i, (n, m) in enumerate(zip(near_pct, major_pct)):
    total = n + m
    if total > 0:
        ax.text(i, total + 0.5, f"{total:.1f}%", ha="center", fontsize=10, fontweight="bold")

ax.set_ylabel("Disagreement (%)")
ax.set_title("Rater Disagreement per Ablation Method\n(near-miss vs major)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("chart_disagreement_breakdown.png", dpi=150)
plt.show()

In [ ]:
#List item disagreement
all_disagreements = pd.concat(
    [get_disagreement_items(c, r1, r2) for c in ABLATION_COLS],
    ignore_index=True
)

print(f"Total item disagreement (all metode): {len(all_disagreements)}")
print(all_disagreements["Severity"].value_counts())

all_disagreements.to_csv("disagreement_items_for_review.csv", index=False)
all_disagreements.head(10)

In [ ]:
overall_n = int(summary["N"].sum())
overall_disagree = int(summary["Disagree"].sum())
overall_pct = round(overall_disagree / overall_n * 100, 2)

overall_near = int(summary["Near_miss (1-level)"].sum())
overall_major = int(summary["Major (2-level)"].sum())

print(f"Total item       : {overall_n}")
print(f"Total disagreement      : {overall_disagree} ({overall_pct}%)")
print(f"  - Near-miss (1 level) : {overall_near} ({round(overall_near/overall_n*100,2)}%)")
print(f"  - Major (2 level)     : {overall_major} ({round(overall_major/overall_n*100,2)}%)")
print(f"Total agreement         : {overall_n - overall_disagree} ({round((overall_n-overall_disagree)/overall_n*100,2)}%)")